# Visualising Architectures and Quantization

## Importing Hugging Face Access Token

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

## Logging in to Hugging Face

In [ ]:
from huggingface_hub import login

In [ ]:
login(HF_TOKEN)

## Installing the Libraries

In [ ]:
!pip install -q bitsandbytes accelerate

## Importing the Libraries

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig

## Defining Models and Message

In [ ]:
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke that will resonate with a room full of data scientists."}
]

## Quantization Configurations

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_quant_type = "nf4"
)

## Loading LLAMA's Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors = "pt"
).to("cuda")

In [ ]:
print(inputs)

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   2148,  10263,    220,   2366,     21,    271,   2675,    527,
            264,  11190,  18328, 128009, 128006,    882, 128007,    271,  41551,
            264,   3177,  70395,  22380,    430,    690,  89986,    449,    264,
           3130,   2539,    315,    828,  14248,     13, 128009]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1]], device='cuda:0')}


## Loading the model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map = "auto",
    quantization_config = quantization_config
)

In [ ]:
model.generation_config.pad_token_id = tokenizer.pad_token_id

## Printing Memory Footprint

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory Footprint: {memory:,.2f} MB")

Memory Footprint: 1,012.01 MB


## Inspecting LLAMA's Internal

In [ ]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

## Generating New Tokens

In [ ]:
encoded_output = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 80
)

In [ ]:
print(encoded_output)

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   2148,  10263,    220,   2366,     21,    271,   2675,    527,
            264,  11190,  18328, 128009, 128006,    882, 128007,    271,  41551,
            264,   3177,  70395,  22380,    430,    690,  89986,    449,    264,
           3130,   2539,    315,    828,  14248,     13, 128009, 128006,  78191,
         128007,    271,     32,    828,  28568,  23291,   1139,    264,   6875,
            323,  17501,    279,  95307,     11,    330,   5519,    499,    617,
            904,   6603,    389,  21075,  11478,   7673,   4815,    791,  95307,
          31737,     11,    330,   2181,    596,    264,   5780,   6975,   1210,
         128009]], device='cuda:0')


In [ ]:
decoded_output = tokenizer.decode(encoded_output[0])

In [ ]:
print(decoded_output)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 31 Jul 2026

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke that will resonate with a room full of data scientists.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

A data scientist walks into a library and asks the librarian, "Do you have any books on artificial intelligence?" 

The librarian replies, "It's a machine learning."<|eot_id|>


## Memory Cleaning

In [ ]:
del model, inputs, tokenizer, encoded_output, decoded_output
gc.collect()
torch.cuda.empty_cache()

## Creating Generate Function

In [ ]:
def generate(model, messages, quantization = True, max_new_token = 80):

  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token

  inputs = tokenizer.apply_chat_template(
      messages,
      return_tensors = "pt",
      add_generation_prompt = True
  ).to("cuda")

  streamer = TextStreamer(tokenizer)

  if quantization:
    model = AutoModelForCausalLM.from_pretrained(
        model,
        quantization_config = quantization_config
    ).to("cuda")

  else:
    model = AutoModelForCausalLM.from_pretrained(
        model,
    ).to("cuda")

  outputs = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_new_tokens = max_new_token,
      streamer = streamer
  )

  return tokenizer.decode(outputs[0])

## Using PHI

In [ ]:
output_PHI = generate(PHI, messages)

In [ ]:
print(output_PHI)

<|system|>You are a helpful assistant<|end|><|user|>Tell a light-hearted joke that will resonate with a room full of data scientists.<|end|><|assistant|>Sure, here's a joke tailored for data scientists:

Why did the data scientist break up with their computer?

Because it kept asking for more space, and they just couldn't handle the "big data"!<|end|>


## Using Gemma

In [ ]:
output_GEMMA = generate(GEMMA, messages, quantization = False)

In [ ]:
print(output_GEMMA)

<bos><start_of_turn>user
You are a helpful assistant

Tell a light-hearted joke that will resonate with a room full of data scientists.<end_of_turn>
<start_of_turn>model
Why don't scientists ever have a TV?

Because they're too busy **plumbing**! <end_of_turn>


## Using QWEN

In [ ]:
output_QWEN = generate(QWEN, messages)

In [ ]:
print(output_QWEN)

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Tell a light-hearted joke that will resonate with a room full of data scientists.<|im_end|>
<|im_start|>assistant
Sure! Here's a light-hearted data science joke that’ll make even the most stubborn model chuckle:

---

Why did the data scientist bring a ladder to the conference?

Because they heard the *regression* was going to be *on the up-and-up* — and they wanted to *predict* the future from the ground up! 📊😄

*(Bonus points if you catch the


## Using DeepSeek

In [ ]:
output_DEEPSEEK = generate(DEEPSEEK, messages, quantization = False)

In [ ]:
print(output_DEEPSEEK)

<｜begin▁of▁sentence｜>You are a helpful assistant<｜User｜>Tell a light-hearted joke that will resonate with a room full of data scientists.<｜Assistant｜><think>
Alright, I need to come up with a light-hearted joke that will resonate with a room full of data scientists. Hmm, data scientists love statistics, numbers, and patterns. They often joke about how they're not really human, but they're the best at analyzing data.

So, the audience is data scientists. They're probably into numbers, data, maybe even a bit of humor. The joke
